# 1주차 API 연동 테스트

## 목표
- 더미 데이터(가상의 분석 결과값)를 설계한다
- 더미 데이터를 프롬프트로 변환한다
- Gemini API를 호출하여 자연어 설명이 출력되는지 확인한다

## 흐름
더미 데이터 → 프롬프트 빌더 → Gemini API 호출 → 자연어 출력 확인

In [2]:
import os
from dotenv import load_dotenv
from google import genai

# .env 파일에서 API 키 로드
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# API 키 정상 로드 확인 (키 전체를 출력하지 않도록 앞 5자리만 확인)
print(f"API 키 로드 확인: {GEMINI_API_KEY[:5]}...")

API 키 로드 확인: AIzaS...


## 더미 데이터 설계

실제 분석 모델링(3주차)이 완성되면 아래 딕셔너리 형태의 값이 자동으로 산출된다.
지금은 그 결과값을 손으로 흉내내어 API 테스트에 활용한다.

포함 항목:
- 종목 기본 정보 (종목명, 현재가)
- 기술적 지표 (RSI, 이동평균 대비 현재가)
- 거래량 이상치 여부 (Z-score 기반)
- 외국인·기관 수급 방향
- 뉴스 감성 점수

In [3]:
# 더미 데이터 — 3주차 분석 모델링이 산출할 결과값을 흉내낸 딕셔너리
dummy_data = {
    "종목명": "삼성전자",
    "현재가": 75000,
    "RSI": 72.3,                      # 70 이상 = 과열 구간
    "이동평균_대비": "상회",            # 현재가가 20일 이동평균선 위에 있음
    "거래량_이상": True,               # Z-score 2σ 초과 → 이상 거래량 감지
    "거래량_배율": 2.1,                # 평소 대비 2.1배
    "외국인_순매수": "매수",
    "기관_순매수": "매도",
    "뉴스_감성": "긍정",
    "뉴스_감성_점수": 0.74             # 1에 가까울수록 긍정
}

print(dummy_data)

{'종목명': '삼성전자', '현재가': 75000, 'RSI': 72.3, '이동평균_대비': '상회', '거래량_이상': True, '거래량_배율': 2.1, '외국인_순매수': '매수', '기관_순매수': '매도', '뉴스_감성': '긍정', '뉴스_감성_점수': 0.74}


## 프롬프트 빌더

분석 결과 딕셔너리를 Gemini API에 전달할 프롬프트 문자열로 변환한다.

설계 원칙:
- LLM은 수치를 직접 생성하지 않는다 (환각 리스크 방지)
- 수치는 반드시 더미 데이터에서 가져온다
- 초보 투자자 눈높이의 자연어 설명을 요청한다
- 면책 고지를 프롬프트에 포함시킨다

In [4]:
def build_prompt(data: dict) -> str:
    """
    분석 결과 딕셔너리를 받아 Gemini API용 프롬프트 문자열을 반환한다.
    LLM이 수치를 생성하지 않도록 모든 수치는 data에서 직접 삽입한다.
    """
    prompt = f"""
당신은 주식 초보 투자자를 위한 친절한 AI 분석 도우미입니다.
아래 분석 데이터를 바탕으로 초보 투자자가 이해할 수 있는 쉬운 말로 설명해주세요.

[분석 데이터]
- 종목명: {data['종목명']}
- 현재가: {data['현재가']:,}원
- RSI: {data['RSI']} (70 이상은 과열 구간, 30 이하는 과매도 구간)
- 이동평균 대비: 현재가가 20일 이동평균선을 {data['이동평균_대비']}
- 거래량 이상 감지: {data['거래량_이상']} (평소 대비 {data['거래량_배율']}배)
- 외국인 순매수: {data['외국인_순매수']}
- 기관 순매수: {data['기관_순매수']}
- 뉴스 감성: {data['뉴스_감성']} (감성 점수: {data['뉴스_감성_점수']})

[출력 형식]
3~5문장으로 요약해주세요.
전문 용어는 괄호 안에 간단한 설명을 덧붙여주세요.
마지막 문장은 반드시 아래 면책 고지로 끝내주세요.

[면책 고지]
본 내용은 투자 참고 정보이며 투자 권유가 아닙니다. 투자 판단과 책임은 투자자 본인에게 있습니다.
"""
    return prompt

# 프롬프트 확인
prompt = build_prompt(dummy_data)
print(prompt)


당신은 주식 초보 투자자를 위한 친절한 AI 분석 도우미입니다.
아래 분석 데이터를 바탕으로 초보 투자자가 이해할 수 있는 쉬운 말로 설명해주세요.

[분석 데이터]
- 종목명: 삼성전자
- 현재가: 75,000원
- RSI: 72.3 (70 이상은 과열 구간, 30 이하는 과매도 구간)
- 이동평균 대비: 현재가가 20일 이동평균선을 상회
- 거래량 이상 감지: True (평소 대비 2.1배)
- 외국인 순매수: 매수
- 기관 순매수: 매도
- 뉴스 감성: 긍정 (감성 점수: 0.74)

[출력 형식]
3~5문장으로 요약해주세요.
전문 용어는 괄호 안에 간단한 설명을 덧붙여주세요.
마지막 문장은 반드시 아래 면책 고지로 끝내주세요.

[면책 고지]
본 내용은 투자 참고 정보이며 투자 권유가 아닙니다. 투자 판단과 책임은 투자자 본인에게 있습니다.



## Gemini API 호출 및 출력 확인

모델 전략:
- 단순 설명 (시장 브리핑, 수급 요약) → gemini-2.5-flash-lite
- 핵심 설명 (차트 해석, Q&A)        → gemini-2.5-flash

1주차 테스트는 핵심 설명에 해당하므로 gemini-2.5-flash로 호출한다.

In [6]:
# 모델 설정
# 단순 설명용: gemini-2.5-flash-lite
# 핵심 설명용: gemini-2.5-flash
FLASH_MODEL = "gemini-2.5-flash"
FLASH_LITE_MODEL = "gemini-2.5-flash-lite"

# Gemini API 클라이언트 설정
client = genai.Client(api_key=GEMINI_API_KEY)

# 1주차 테스트 → 핵심 설명 모델(gemini-2.5-flash)로 호출
response = client.models.generate_content(
    model=FLASH_MODEL,
    contents=prompt
)

# 출력 확인
print("=== Gemini API 출력 결과 ===")
print(response.text)

=== Gemini API 출력 결과 ===
삼성전자의 현재 가격은 75,000원이며, 주식 시장의 여러 지표를 통해 현재 상황을 파악할 수 있습니다. RSI(상대강도지수, 주가의 매수/매도 압력을 나타내는 지표)가 72.3으로 현재 주가가 다소 과열(너무 많이 올랐다는 뜻)되어 있다는 신호를 보냅니다. 하지만 현재가가 20일 이동평균선(지난 20일 동안의 평균 주가) 위에 있어 단기적인 상승세에 있으며, 평소보다 2.1배 많은 거래량(주식이 사고팔린 양)이 감지되어 주식에 대한 관심이 높습니다. 외국인 투자자들은 순매수했지만 기관 투자자들은 순매도하는 상반된 모습이며, 뉴스 감성 역시 전반적으로 긍정적인 분위기입니다.

본 내용은 투자 참고 정보이며 투자 권유가 아닙니다. 투자 판단과 책임은 투자자 본인에게 있습니다.
